# ADAN0 Trading Bot — Full GPU Training Pipeline

**Target**: Colab H100 (12GB RAM, 80GB VRAM) or A100, or Kaggle P100

**Pipeline**:
1. Setup & Dependencies
2. Download 6 years BTC/USDT data via CCXT (Bitget/OKX)
3. Compute 21 technical indicators per timeframe
4. Create train/test/val splits (70/20/10)
5. Launch 4-worker PBT training (500K+ steps)
6. Deterministic backtest on TEST split (OOS)
7. Export model for paper trading

**S15 Hard Reset**: Pure realized PnL reward. No shaping. No bonuses.

---

## 0. GPU Verification & Resource Check

In [ ]:
import subprocess, sys, os, platform

# GPU check
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                           '--format=csv,noheader'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    gpu_name, vram, driver = gpu_info.stdout.strip().split(', ')
    print(f"\u2705 GPU: {gpu_name}")
    print(f"   VRAM: {vram}")
    print(f"   Driver: {driver}")
else:
    print("\u274c NO GPU detected! This notebook requires a GPU runtime.")
    print("   Go to: Runtime > Change runtime type > GPU (H100 or A100)")
    raise RuntimeError("GPU required")

# RAM check
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"\u2705 RAM: {ram_gb:.1f} GB")
print(f"\u2705 CPUs: {os.cpu_count()}")
print(f"\u2705 Platform: {platform.platform()}")

# Determine training intensity based on resources
if ram_gb >= 80:
    TRAINING_STEPS = 1_000_000
    NUM_WORKERS = 4
    print(f"\n\U0001f680 HIGH-END CONFIG: {TRAINING_STEPS:,} steps, {NUM_WORKERS} workers")
elif ram_gb >= 12:
    TRAINING_STEPS = 500_000
    NUM_WORKERS = 4
    print(f"\n\U0001f680 STANDARD CONFIG: {TRAINING_STEPS:,} steps, {NUM_WORKERS} workers")
else:
    TRAINING_STEPS = 200_000
    NUM_WORKERS = 2
    print(f"\n\u26a0\ufe0f LOW-RAM CONFIG: {TRAINING_STEPS:,} steps, {NUM_WORKERS} workers")

## 1. Setup — Clone Repository & Install Dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Cabrel10/ADAN0.git"
BRANCH = "genspark_ai_developer"
WORK_DIR = "/content/ADAN0"

# Clone or update
if os.path.exists(WORK_DIR):
    print("Repository exists, pulling latest...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR], check=True)
    os.chdir(WORK_DIR)

print(f"\u2705 Working directory: {os.getcwd()}")
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

In [ ]:
%%capture install_output
# Install dependencies without breaking Colab's pre-installed packages
import subprocess, sys

DEPS = [
    "ccxt>=4.0",
    "stable-baselines3[extra]>=2.3",
    "ray[tune]>=2.9",
    "pandas>=2.0",
    "numpy>=1.24",
    "pandas-ta>=0.3.14b",
    "scikit-learn>=1.3",
    "PyYAML>=6.0",
    "rich>=13.0",
    "tqdm>=4.64",
    "joblib>=1.3",
    "gymnasium>=0.29",
    "tensorboard>=2.14",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps"] + DEPS,
    check=False  # Don't fail on individual package issues
)

# Verify critical imports
import importlib
for pkg in ['ccxt', 'stable_baselines3', 'ray', 'pandas_ta', 'sklearn']:
    try:
        importlib.import_module(pkg)
        print(f"\u2705 {pkg}")
    except ImportError as e:
        print(f"\u274c {pkg}: {e}")
        # Retry with deps
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

In [ ]:
# Show install output if there were issues
output = install_output.stdout + install_output.stderr
errors = [l for l in output.split('\n') if 'ERROR' in l or 'error' in l.lower()]
if errors:
    print("\u26a0\ufe0f Install warnings:")
    for e in errors[:10]:
        print(f"  {e}")
else:
    print("\u2705 All dependencies installed cleanly")

# Set PYTHONPATH
import sys
sys.path.insert(0, '/content/ADAN0/src')
os.environ['PYTHONPATH'] = '/content/ADAN0/src:' + os.environ.get('PYTHONPATH', '')
print(f"\u2705 PYTHONPATH configured")

## 2. Download 6 Years of BTC/USDT Data via CCXT

Downloads from Bitget (primary) or OKX (fallback):
- **5m**: ~630,000 candles (6 years)
- **1h**: ~52,560 candles
- **4h**: ~13,140 candles

This takes 10-20 minutes due to exchange rate limits.

In [ ]:
import ccxt
import pandas as pd
import numpy as np
import time
import logging
from datetime import datetime, timezone, timedelta
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('data_download')

# ============================================================================
# CONFIGURATION
# ============================================================================
SYMBOL = "BTC/USDT"
YEARS_BACK = 6  # 6 years of history
OUTPUT_BASE = Path("/content/ADAN0/data/raw/BTCUSDT")
EXCHANGES = ["bitget", "okx", "kucoin", "bybit"]

# Timeframe configs with target candle counts
TIMEFRAMES = {
    "5m":  {"minutes": 5,   "target": min(630_000, YEARS_BACK * 365 * 24 * 12)},  # 6 years
    "1h":  {"minutes": 60,  "target": min(52_560, YEARS_BACK * 365 * 24)},
    "4h":  {"minutes": 240, "target": min(13_140, YEARS_BACK * 365 * 6)},
}

def connect_exchange():
    """Connect to first working exchange with BTC/USDT."""
    for name in EXCHANGES:
        try:
            ex_cls = getattr(ccxt, name)
            ex = ex_cls({
                'enableRateLimit': True,
                'options': {'defaultType': 'spot'},
                'timeout': 30000,
            })
            ex.load_markets()
            if SYMBOL not in ex.markets:
                logger.warning(f"{name}: {SYMBOL} not available")
                continue
            # Verify with test fetch
            test = ex.fetch_ohlcv(SYMBOL, '1h', limit=5)
            if test and len(test) >= 3:
                logger.info(f"\u2705 Connected to {name.upper()} ({len(ex.markets)} markets)")
                return ex, name
        except Exception as e:
            logger.warning(f"\u274c {name}: {str(e)[:100]}")
    raise RuntimeError("Cannot connect to any exchange. Check network.")


def download_full_history(exchange, tf_name, tf_config):
    """Download full history with pagination. Handles rate limits gracefully."""
    target = tf_config['target']
    minutes = tf_config['minutes']
    
    # Start from YEARS_BACK years ago
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=YEARS_BACK * 365)
    since_ms = int(start_dt.timestamp() * 1000)
    end_ms = int(end_dt.timestamp() * 1000)
    
    logger.info(f"\n{'='*60}")
    logger.info(f"Downloading {SYMBOL} {tf_name} | {start_dt.date()} \u2192 {end_dt.date()}")
    logger.info(f"Target: ~{target:,} candles")
    
    all_data = []
    batch_count = 0
    max_retries = 5
    
    while since_ms < end_ms:
        for retry in range(max_retries):
            try:
                ohlcv = exchange.fetch_ohlcv(SYMBOL, tf_name, since=since_ms, limit=1000)
                if not ohlcv:
                    since_ms = end_ms  # Exit loop
                    break
                
                all_data.extend(ohlcv)
                batch_count += 1
                since_ms = ohlcv[-1][0] + 1  # Next batch starts after last candle
                
                if batch_count % 20 == 0:
                    pct = len(all_data) / target * 100
                    logger.info(f"  [{tf_name}] {len(all_data):,} candles ({pct:.1f}%) | batch {batch_count}")
                
                if len(ohlcv) < 100:  # End of available data
                    since_ms = end_ms
                    break
                
                time.sleep(exchange.rateLimit / 1000)  # Respect rate limit
                break  # Success, exit retry loop
                
            except ccxt.RateLimitExceeded:
                wait = 5 * (retry + 1)
                logger.warning(f"  Rate limit hit, waiting {wait}s (retry {retry+1}/{max_retries})")
                time.sleep(wait)
            except (ccxt.NetworkError, ccxt.ExchangeNotAvailable) as e:
                wait = 10 * (retry + 1)
                logger.warning(f"  Network error: {str(e)[:50]}, waiting {wait}s")
                time.sleep(wait)
            except Exception as e:
                logger.error(f"  Fatal error: {e}")
                if retry == max_retries - 1:
                    break
                time.sleep(5)
    
    if not all_data:
        raise RuntimeError(f"No data downloaded for {tf_name}")
    
    # Build DataFrame
    df = pd.DataFrame(all_data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    df = df.set_index('timestamp')
    df.index = df.index.tz_localize(None)  # Remove timezone for compatibility
    
    # Validate
    vol_usd = (df['volume'] * df['close']).median()
    logger.info(f"  \u2705 {tf_name}: {len(df):,} candles | {df.index[0]} \u2192 {df.index[-1]}")
    logger.info(f"     Median volume: ${vol_usd:,.0f} | Span: {(df.index[-1]-df.index[0]).days} days")
    
    # Save
    out_dir = OUTPUT_BASE / tf_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"BTCUSDT_{tf_name}_raw.parquet"
    df.to_parquet(out_path, engine='pyarrow')
    logger.info(f"     Saved: {out_path} ({out_path.stat().st_size / 1024:.0f} KB)")
    
    return df


# ============================================================================
# EXECUTE DOWNLOAD
# ============================================================================
exchange, exchange_name = connect_exchange()
downloaded = {}

for tf_name, tf_config in TIMEFRAMES.items():
    try:
        df = download_full_history(exchange, tf_name, tf_config)
        downloaded[tf_name] = df
    except Exception as e:
        logger.error(f"\u274c FAILED {tf_name}: {e}")
        raise

print(f"\n{'='*60}")
print(f"\u2705 DOWNLOAD COMPLETE \u2014 {exchange_name.upper()}")
for tf, df in downloaded.items():
    print(f"   {tf}: {len(df):,} candles ({(df.index[-1]-df.index[0]).days} days)")
print(f"{'='*60}")

## 3. Compute Technical Indicators (21 per Timeframe)

Features: OHLCV + 16 technical indicators = 21 columns per TF:
- EMA ratio, MACD histogram, RSI, ADX, DI delta
- ATR%, Bollinger %B, OBV slope, Volume ratio
- Volatility ratio, Fibonacci ratio, Price action
- VWAP ratio, Market structure, BB width, Log return

In [ ]:
import subprocess, sys, os
os.chdir('/content/ADAN0')

result = subprocess.run(
    [sys.executable, 'scripts/compute_features_real.py'],
    env={**os.environ, 'PYTHONPATH': '/content/ADAN0/src'},
    capture_output=True, text=True, timeout=300
)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(f"\n\u274c STDERR:\n{result.stderr[-2000:]}")
    raise RuntimeError("Feature computation failed")
else:
    print("\n\u2705 Features computed successfully (21 columns per TF)")

## 4. Create Train/Test/Val Splits (70/20/10)

Chronological splits (no leakage):
- **Train** (70%): Model learning
- **Test** (20%): OOS backtest evaluation  
- **Val** (10%): Hyperparameter tuning validation

In [ ]:
import subprocess, sys, os
os.chdir('/content/ADAN0')

result = subprocess.run(
    [sys.executable, 'scripts/create_train_test_val_splits.py'],
    env={**os.environ, 'PYTHONPATH': '/content/ADAN0/src'},
    capture_output=True, text=True, timeout=120
)

print(result.stdout)
if result.returncode != 0:
    print(f"\u274c Error: {result.stderr}")
    raise RuntimeError("Split creation failed")

# Verify splits
import pandas as pd
from pathlib import Path

print("\n" + "="*60)
print("SPLIT VERIFICATION:")
print("="*60)
for split in ['train', 'test', 'val']:
    for tf in ['5m', '1h', '4h']:
        path = Path(f'/content/ADAN0/data/processed/indicators/{split}/BTCUSDT/{tf}.parquet')
        if path.exists():
            df = pd.read_parquet(path)
            print(f"  {split}/{tf}: {len(df):>8,} rows | {df.index[0]} \u2192 {df.index[-1]} | {len(df.columns)} cols")
        else:
            print(f"  \u274c MISSING: {path}")
            raise FileNotFoundError(f"Split file missing: {path}")

print("\n\u2705 All splits created and verified")

## 5. Training — 4-Worker PBT on GPU

**Architecture**:
- 4 workers: Scalper(5m), Intraday(1h), Swing(4h), Position(4h)
- Ray Tune PBT (Population-Based Training)
- PPO with gSDE (State-Dependent Exploration)
- ContextualTemporalFusionExtractor (CNN + Attention)

**Reward**: `symlog(realized_pnl_scaled - costs - drawdown + time_decay)`

**Expected duration**: 1-3 hours depending on GPU and step count.

In [ ]:
import subprocess, sys, os, time
os.chdir('/content/ADAN0')

# Training configuration
STEPS = TRAINING_STEPS  # Set in cell 0 based on resources
INTERVAL = 10_000       # PBT perturbation interval
PROFILES = ["scalper", "intraday", "swing", "position"]
CHECKPOINT_DIR = "/content/ADAN0/training_output/ray_results"

print(f"\U0001f680 LAUNCHING TRAINING")
print(f"   Steps: {STEPS:,}")
print(f"   Workers: {NUM_WORKERS}")
print(f"   Profiles: {', '.join(PROFILES)}")
print(f"   Interval: {INTERVAL:,} steps/PBT iteration")
print(f"   Checkpoint: {CHECKPOINT_DIR}")
print(f"\n{'='*60}")

cmd = [
    sys.executable, 'scripts/train_parallel_agents.py',
    '--mode', 'heavy',
    '--steps', str(STEPS),
    '--steps-per-iter', str(INTERVAL),
    '--num-samples', str(NUM_WORKERS),
    '--num-cpus', str(max(2, os.cpu_count() - 1)),
    '--envs-per-worker', '1',
    '--no-subproc',
    '--profiles', *PROFILES,
    '--checkpoint-dir', CHECKPOINT_DIR,
]

print(f"CMD: {' '.join(cmd)}\n")

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    env={**os.environ, 'PYTHONPATH': '/content/ADAN0/src'},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream output with progress tracking
lines_seen = 0
last_progress = time.time()
for line in iter(proc.stdout.readline, ''):
    lines_seen += 1
    # Show key progress lines
    if any(kw in line for kw in ['timesteps_total', 'mean_reward', 'PBT', 'COMPLETE',
                                   'Worker', 'Checkpoint', 'ERROR', 'FATAL', 'OOM']):
        print(line.rstrip())
    elif time.time() - last_progress > 60:  # Show a heartbeat every 60s
        elapsed = time.time() - t0
        print(f"  ... {elapsed/60:.1f} min elapsed, {lines_seen} log lines ...")
        last_progress = time.time()

proc.wait()
elapsed = time.time() - t0

print(f"\n{'='*60}")
if proc.returncode == 0:
    print(f"\u2705 TRAINING COMPLETE in {elapsed/60:.1f} minutes")
else:
    print(f"\u274c Training exited with code {proc.returncode} after {elapsed/60:.1f} min")
    print("Check logs above for errors.")

## 6. Extract Best Model & Run OOS Backtest

In [ ]:
import json, glob, os
from pathlib import Path

os.chdir('/content/ADAN0')
RESULTS_DIR = Path(CHECKPOINT_DIR)

# Find PBT summary
summary_path = RESULTS_DIR / 'pbt_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print("\U0001f3c6 PBT TRAINING SUMMARY:")
    print(json.dumps(summary, indent=2))
else:
    print("\u26a0\ufe0f No PBT summary found. Looking for checkpoints directly...")

# Find best checkpoint (most recent with highest timesteps)
ckpt_dirs = sorted(glob.glob(str(RESULTS_DIR / '**' / 'checkpoint_*'), recursive=True))
if not ckpt_dirs:
    # Fallback to sandbox checkpoints
    ckpt_dirs = sorted(glob.glob('checkpoints/ppo_adan0_sandbox_*.zip'))

print(f"\nFound {len(ckpt_dirs)} checkpoint(s)")
for d in ckpt_dirs[-5:]:
    print(f"  {d}")

# Use the latest checkpoint
if ckpt_dirs:
    best_ckpt_dir = ckpt_dirs[-1]
    # Find model.zip inside
    if os.path.isdir(best_ckpt_dir):
        model_path = os.path.join(best_ckpt_dir, 'model.zip')
    else:
        model_path = best_ckpt_dir  # It's the zip file itself
    print(f"\n\u2705 Best model: {model_path}")
    print(f"   Size: {os.path.getsize(model_path) / 1024:.0f} KB")
else:
    raise FileNotFoundError("No checkpoints found! Training may have failed.")

In [ ]:
# Run deterministic backtest on TEST split (OOS)
import subprocess, sys, json

os.chdir('/content/ADAN0')

# Determine vecnorm path
vecnorm_path = model_path.replace('.zip', '_vecnorm.pkl') if model_path.endswith('.zip') \
    else os.path.join(os.path.dirname(model_path), 'vecnormalize.pkl')

backtest_cmd = [
    sys.executable, 'scripts/deterministic_backtest.py',
    '--steps', '2000',
    '--split', 'test',
    '--ckpt', model_path,
    '--out', 'logs/validation/colab_oos_backtest.json'
]

print(f"Running OOS backtest on TEST split...")
print(f"CMD: {' '.join(backtest_cmd)}")

result = subprocess.run(
    backtest_cmd,
    env={**os.environ, 'PYTHONPATH': '/content/ADAN0/src'},
    capture_output=True, text=True, timeout=600
)

if result.stdout.strip():
    try:
        backtest_result = json.loads(result.stdout)
        print("\n" + "="*60)
        print("\U0001f4ca HONEST OOS BACKTEST RESULTS")
        print("="*60)
        print(json.dumps(backtest_result, indent=2))
        print("\n" + "="*60)
        print(f"VERDICT: {backtest_result.get('verdict', 'UNKNOWN')}")
        print("="*60)
    except json.JSONDecodeError:
        print(result.stdout[-2000:])

if result.returncode != 0:
    print(f"\n\u26a0\ufe0f stderr: {result.stderr[-1000:]}")

## 7. Export Model for Paper Trading

Exports ONLY what's needed for live paper trading:
- `model.zip` — PPO policy weights
- `vecnormalize.pkl` — Observation normalization stats (if used)
- `config.yaml` — Environment configuration
- `exog_oracle.pkl` — HMM regime oracle
- `worker_state.json` — Training metadata

Download the archive and deploy to your paper trading server.

In [ ]:
import shutil, os, json
from pathlib import Path
from datetime import datetime

os.chdir('/content/ADAN0')

# Create export package
EXPORT_DIR = Path('/content/adan0_paper_trading_package')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Copy model
shutil.copy2(model_path, EXPORT_DIR / 'model.zip')
print(f"\u2705 model.zip ({os.path.getsize(model_path)/1024:.0f} KB)")

# 2. Copy vecnormalize if exists
if os.path.exists(vecnorm_path):
    shutil.copy2(vecnorm_path, EXPORT_DIR / 'vecnormalize.pkl')
    print(f"\u2705 vecnormalize.pkl")
else:
    print(f"\u26a0\ufe0f vecnormalize.pkl not found (VecNormalize may be disabled)")

# 3. Copy config
shutil.copy2('config/config.yaml', EXPORT_DIR / 'config.yaml')
print(f"\u2705 config.yaml")

# 4. Copy oracle model
oracle_path = 'models/exog_oracle.pkl'
if os.path.exists(oracle_path):
    shutil.copy2(oracle_path, EXPORT_DIR / 'exog_oracle.pkl')
    print(f"\u2705 exog_oracle.pkl")

# 5. Create deployment manifest
manifest = {
    'created': datetime.now().isoformat(),
    'model_path': str(model_path),
    'training_steps': TRAINING_STEPS,
    'num_workers': NUM_WORKERS,
    'reward_formula': 'symlog(realized_pnl_scaled - trade_cost - drawdown + time_decay)',
    'reward_shaping': 'NONE (S15 Hard Reset)',
    'backtest_split': 'test (OOS)',
    'exchange_target': ['bitget', 'binance'],
    'initial_capital': 20.50,
    'deployment_mode': 'paper_trading',
    'files': list(str(p.name) for p in EXPORT_DIR.iterdir()),
}
with open(EXPORT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"\u2705 manifest.json")

# 6. Create tar archive
archive_name = f'adan0_paper_trading_{datetime.now().strftime("%Y%m%d_%H%M")}'
shutil.make_archive(f'/content/{archive_name}', 'gztar', EXPORT_DIR)
archive_path = f'/content/{archive_name}.tar.gz'
print(f"\n{'='*60}")
print(f"\U0001f4e6 EXPORT PACKAGE READY")
print(f"   Path: {archive_path}")
print(f"   Size: {os.path.getsize(archive_path)/1024:.0f} KB")
print(f"{'='*60}")
print(f"\nContents:")
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name:30s} {f.stat().st_size/1024:>8.0f} KB")

In [ ]:
# Download to local machine (Colab)
try:
    from google.colab import files
    print("\U0001f4e5 Downloading archive to your machine...")
    files.download(archive_path)
    print("\u2705 Download started. Check your browser downloads.")
except ImportError:
    print("Not running on Colab. Archive saved at:")
    print(f"  {archive_path}")
    print("\nTo use on Kaggle, upload to Kaggle Datasets or use kaggle API.")

## 8. Training Diagnostics & Honest Assessment

**Critical metrics to check before going live:**
- `explained_variance > 0` → Critic is learning
- `win_rate > 50%` on OOS data → Edge exists
- `max_drawdown < 20%` → Risk is controlled
- `total_trades > 50` → Statistical significance

**If ANY metric fails: DO NOT deploy to production.**

In [ ]:
import json
from pathlib import Path

os.chdir('/content/ADAN0')

# Load backtest results
bt_path = Path('logs/validation/colab_oos_backtest.json')
if bt_path.exists():
    with open(bt_path) as f:
        bt = json.load(f)
else:
    print("\u274c No backtest results found")
    bt = {}

# Production readiness check
print("\n" + "="*60)
print("\U0001f6a8 PRODUCTION READINESS ASSESSMENT")
print("="*60)

checks = []
total_trades = bt.get('env_total_trades', 0)
win_rate = bt.get('env_winning_trades', 0) / max(total_trades, 1)
drawdown = bt.get('env_drawdown_pct', 100)
total_return = bt.get('total_return_pct', -100)

checks.append(('Total trades > 50', total_trades > 50, f'{total_trades} trades'))
checks.append(('Win rate > 50%', win_rate > 0.5, f'{win_rate*100:.1f}%'))
checks.append(('Max drawdown < 20%', drawdown < 20, f'{drawdown:.1f}%'))
checks.append(('OOS return > 0%', total_return > 0, f'{total_return:+.2f}%'))
checks.append(('Model file exists', os.path.exists(model_path), str(model_path)[-40:]))

all_pass = True
for name, passed, detail in checks:
    status = '\u2705' if passed else '\u274c'
    print(f"  {status} {name:30s} [{detail}]")
    if not passed:
        all_pass = False

print("\n" + "-"*60)
if all_pass:
    print("\U0001f7e2 ALL CHECKS PASSED \u2014 Model is a candidate for paper trading")
    print("   Deploy with SMALL capital first. Monitor for 2 weeks minimum.")
else:
    print("\U0001f534 CHECKS FAILED \u2014 DO NOT DEPLOY")
    print("   Train longer (1M+ steps) or review the reward/data pipeline.")
    print("   This is NORMAL for under-trained models. Not a bug.")
print("-"*60)